# XOR and Linear Limitations

Companion notebook for Section 1 of the MLP lecture notes.

Objectives:

- visualize why XOR is not linearly separable;
- compare logistic regression with a small MLP;
- show that hidden layers learn a representation in which a nonlinear problem becomes easier;
- connect XOR with other nonlinear classification patterns such as concentric circles.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


## 1. The XOR dataset

The XOR function returns 1 when exactly one of the two binary inputs is 1. No single straight line separates the positive and negative examples.


In [ ]:
X_xor = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
y_xor = np.array([0, 1, 1, 0])

pd.DataFrame(X_xor, columns=['x1', 'x2']).assign(y=y_xor)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for cls, marker in [(0, 'o'), (1, 's')]:
    mask = y_xor == cls
    ax.scatter(X_xor[mask, 0], X_xor[mask, 1], s=180, marker=marker, label=f'class {cls}')
ax.set_xlim(-0.25, 1.25)
ax.set_ylim(-0.25, 1.25)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('XOR is not linearly separable')
ax.legend()
plt.show()


## 2. Logistic regression fails on XOR

Logistic regression learns a linear decision boundary in the original input space. On XOR, that is the wrong geometry.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

logreg = LogisticRegression(C=1e6, solver='lbfgs')
logreg.fit(X_xor, y_xor)

pred_logreg = logreg.predict(X_xor)
print('Predictions:', pred_logreg)
print('Training accuracy:', accuracy_score(y_xor, pred_logreg))
print('Coefficients:', logreg.coef_)
print('Intercept:', logreg.intercept_)


In [ ]:
def plot_decision_boundary(model, X, y, title, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    x_min, x_max = X[:, 0].min() - 0.4, X[:, 0].max() + 0.4
    y_min, y_max = X[:, 1].min() - 0.4, X[:, 1].max() + 0.4
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
    grid = np.c_[xx.ravel(), yy.ravel()]
    if hasattr(model, 'predict_proba'):
        zz = model.predict_proba(grid)[:, 1]
    else:
        zz = model.predict(grid)
    zz = zz.reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=30, cmap='RdBu_r', alpha=0.65)
    ax.contour(xx, yy, zz, levels=[0.5], colors='black', linewidths=2)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolor='black', s=120)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.set_title(title)
    return ax

plot_decision_boundary(logreg, X_xor, y_xor, 'Logistic regression on XOR')
plt.show()


## 3. A small MLP solves XOR

A hidden layer applies a learned nonlinear transformation. The output layer can then solve a simpler problem in the learned representation.


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

mlp_xor = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(4,),
        activation='tanh',
        solver='lbfgs',
        alpha=0.0,
        max_iter=5000,
        random_state=42,
    ),
)
mlp_xor.fit(X_xor, y_xor)

pred_mlp = mlp_xor.predict(X_xor)
print('Predictions:', pred_mlp)
print('Training accuracy:', accuracy_score(y_xor, pred_mlp))


In [ ]:
plot_decision_boundary(mlp_xor, X_xor, y_xor, 'One-hidden-layer MLP on XOR')
plt.show()


## 4. Inspecting the hidden representation

The hidden layer creates new coordinates. In those coordinates, the output layer can separate the examples even though the original input coordinates were not linearly separable.


In [ ]:
scaler = mlp_xor.named_steps['standardscaler']
net = mlp_xor.named_steps['mlpclassifier']
X_scaled = scaler.transform(X_xor)
hidden = np.tanh(X_scaled @ net.coefs_[0] + net.intercepts_[0])

hidden_df = pd.DataFrame(hidden, columns=[f'h{i+1}' for i in range(hidden.shape[1])]).assign(y=y_xor)
hidden_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap='bwr', edgecolor='black', s=120)
axes[0].set_title('Original input space')
axes[0].set_xlabel('x1')
axes[0].set_ylabel('x2')

axes[1].scatter(hidden[:, 0], hidden[:, 1], c=y_xor, cmap='bwr', edgecolor='black', s=120)
axes[1].set_title('Two hidden activations')
axes[1].set_xlabel('h1')
axes[1].set_ylabel('h2')
plt.tight_layout()
plt.show()


## 5. Concentric circles: another nonlinear pattern

The same idea appears in larger synthetic datasets. A linear model struggles with circular class structure, while an MLP can learn a nonlinear boundary.


In [ ]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

X_circles, y_circles = make_circles(n_samples=600, noise=0.08, factor=0.35, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_circles, y_circles, test_size=0.3, stratify=y_circles, random_state=42
)

circle_logreg = make_pipeline(StandardScaler(), LogisticRegression())
circle_mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', solver='adam',
                  learning_rate_init=0.01, max_iter=2000, random_state=42)
)

circle_logreg.fit(X_train, y_train)
circle_mlp.fit(X_train, y_train)

print('Logistic regression test accuracy:', accuracy_score(y_test, circle_logreg.predict(X_test)))
print('MLP test accuracy:', accuracy_score(y_test, circle_mlp.predict(X_test)))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
plot_decision_boundary(circle_logreg, X_circles, y_circles, 'Linear model on circles', axes[0])
plot_decision_boundary(circle_mlp, X_circles, y_circles, 'MLP on circles', axes[1])
plt.tight_layout()
plt.show()


## Exercises

1. Change the hidden layer of the XOR MLP from 4 neurons to 2 neurons. Does it still solve XOR reliably?
2. Replace `activation='tanh'` with `activation='relu'` in the XOR example. What changes?
3. Increase the noise in `make_circles`. At what point does the MLP begin to overfit or lose accuracy?


## Takeaways

- Linear models need a linearly separable representation.
- Hidden layers can learn nonlinear intermediate representations.
- An MLP can solve XOR and concentric-circle patterns without manually engineered polynomial features.
